# Notebook 03 — ETL & SQL

**Objective:** Build a unified daily time-series by aggregating raw POS data and joining all dimension tables via DuckDB.

**Output:** `data/processed/daily_timeseries.csv`

```
date | covers | revenue | avg_check
     | is_weekend | is_holiday | is_ponte | is_high_season | season
     | avg_temp | rain_mm | is_bad_weather
     | event_magnitude | event_radius_km
     | pun_eur_mwh | energy_cost_index
```

In [1]:
import sys
import pandas as pd
import duckdb
from pathlib import Path

# Add project root and src/ to sys.path
# - project root: needed for 'src.utils_forecast' package import
# - src/: needed because utils.py does 'import utils as ut' internally
_project_root = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / 'src').exists())
for _p in [str(_project_root), str(_project_root / 'src')]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

import utils              as ut    # src/utils.py — existing utilities
import src.utils_forecast as ut_f  # src/utils_forecast.py — forecasting utilities

RAW_POS      = _project_root / 'data/raw/raw_sales_pos.csv'
EXT_PATH     = _project_root / 'data/external'
PROCESSED    = _project_root / 'data/processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

print('Project root:', _project_root)
print('DuckDB version:', duckdb.__version__)

Project root: /Users/giovannilopresti/Documents/Data Analytics/Progetti Risto/horeca_forecasting
DuckDB version: 1.4.3


---
## Phase A — Daily POS Aggregation

Raw POS has one row per item sold. Steps:
1. Parse mixed-format dates (`dd/mm/yyyy HH:MM` and `yyyy-mm-dd HH:MM:SS`) → extract date
2. Normalize `tavolo` (lowercase + strip)
3. **Covers:** deduplicate by `(date, tavolo)` → `groupby(date).sum(coperti)`
4. **Revenue:** exclude `Bevande` → `groupby(date).sum(totale_riga)`
5. **avg_check:** `revenue / covers`

In [2]:
# ── Load raw POS ──────────────────────────────────────────────────────────────
pos_raw = pd.read_csv(RAW_POS)
print(f'Raw POS loaded: {pos_raw.shape[0]:,} rows x {pos_raw.shape[1]} cols')

# ── basic_cleaning: parse dates, strip/lowercase strings, @ → a ──────────────
# mapping_dict handles known label variants; left empty — cleaning is textual only
mapping_dict = {}
pos = ut.basic_cleaning(pos_raw, mapping_dict)

# ── date_accuracy: report rows outside [2023, 2024] ──────────────────────────
print('\nDate accuracy check (year_list=[2023, 2024]):')
ut.date_accuracy(pos, 'data_ora', [2023, 2024])

# ── Filter to 2023-2024 using 'anno' column created by date_accuracy ──────────
pos = pos[pos['anno'].isin([2023, 2024])].copy()
print(f'\nAfter year filter: {len(pos):,} rows')

# ── Extract date (data_ora is now a proper datetime after basic_cleaning) ───────
pos['date'] = pos['data_ora'].dt.date

print(f'Categorie: {sorted(pos["categoria"].dropna().unique())}')
print(f'Tavoli:    {sorted(pos["tavolo"].unique())}')
print(f'Date range: {pos["date"].min()} \u2192 {pos["date"].max()}')
print(f'Null dates: {pos["date"].isna().sum()}')


Raw POS loaded: 64,654 rows x 12 cols

Date accuracy check (year_list=[2023, 2024]):
419 rows out of range found.
[2022 2033 2034 2025]

After year filter: 64,235 rows
Categorie: ['antipasti', 'bevande', 'dolci', 'extra', 'primi', 'secondi']
Tavoli:    ['asporto', 'bar', 't1', 't10', 't11', 't12', 't13', 't14', 't15', 't2', 't3', 't4', 't5', 't6', 't7', 't8', 't9', 'terrazza']
Date range: 2023-01-01 → 2024-12-31
Null dates: 0


In [3]:
# ── Covers aggregation ────────────────────────────────────────────────────────
# Each table visit generates multiple rows (one per item ordered).
# coperti is repeated identically for all rows of the same table visit.
# Deduplicate by (date, tavolo) to count each table visit once,
# then sum coperti per day.
# Note: if a table turns over (lunch + dinner same day), only one seating is
# counted — acceptable approximation for this restaurant's typical usage.
covers_daily = (
    pos
    .drop_duplicates(subset=['date', 'tavolo'])[['date', 'tavolo', 'coperti']]
    .groupby('date', as_index=False)['coperti']
    .sum()
    .rename(columns={'coperti': 'covers'})
)

print(f'covers_daily shape: {covers_daily.shape}')
print(f'covers range: {covers_daily["covers"].min()} – {covers_daily["covers"].max()}')
print(f'covers mean:  {covers_daily["covers"].mean():.1f}')
print(f'\nSample:')
display(covers_daily.head(5))

covers_daily shape: (731, 2)
covers range: 21.0 – 83.0
covers mean:  54.1

Sample:


,date,covers
0,2023-01-01,46.0
1,2023-01-02,42.0
2,2023-01-03,52.0
3,2023-01-04,56.0
4,2023-01-05,56.0


In [4]:
# ── Revenue aggregation (food only — exclude Bevande) ─────────────────────────
food_mask = pos['categoria'] != 'bevande'  # lowercased by basic_cleaning
n_bevande = (~food_mask).sum()
print(f'Rows excluded (Bevande): {n_bevande:,} ({n_bevande/len(pos)*100:.1f}%)')

revenue_daily = (
    pos[food_mask]
    .groupby('date', as_index=False)['totale_riga']
    .sum()
    .rename(columns={'totale_riga': 'revenue'})
)
revenue_daily['revenue'] = revenue_daily['revenue'].round(2)

print(f'revenue_daily shape: {revenue_daily.shape}')
print(f'revenue range: {revenue_daily["revenue"].min():.2f} – {revenue_daily["revenue"].max():.2f} EUR')
print(f'revenue mean:  {revenue_daily["revenue"].mean():.2f} EUR')

Rows excluded (Bevande): 15,918 (24.8%)
revenue_daily shape: (731, 2)
revenue range: 233.21 – 7216.53 EUR
revenue mean:  2003.75 EUR


In [5]:
# ── Merge covers + revenue → daily_pos ───────────────────────────────────────
daily_pos = covers_daily.merge(revenue_daily, on='date', how='inner')
daily_pos['avg_check'] = (daily_pos['revenue'] / daily_pos['covers'])
daily_pos['avg_check'] = daily_pos['avg_check'].replace([float('inf'), float('-inf')], pd.NA).round(2)
daily_pos['date'] = pd.to_datetime(daily_pos['date'])

print(f'daily_pos shape: {daily_pos.shape}')
print(f'NaN covers:   {daily_pos["covers"].isna().sum()}')
print(f'NaN revenue:  {daily_pos["revenue"].isna().sum()}')
print(f'NaN avg_check:{daily_pos["avg_check"].isna().sum()}')
print()
print('Descriptive stats:')
display(daily_pos[['covers', 'revenue', 'avg_check']].describe().round(2))
print('\nFirst 5 rows:')
display(daily_pos.head())

daily_pos shape: (731, 4)
NaN covers:   0
NaN revenue:  0
NaN avg_check:0

Descriptive stats:


,covers,revenue,avg_check
count,731.00,731.00,731.00
mean,54.14,2003.75,37.27
std,8.80,973.22,18.45
min,21.00,233.21,8.94
25%,48.00,1322.60,24.66
50%,54.00,1810.24,33.61
75%,60.00,2485.14,45.57
max,83.00,7216.53,160.87



First 5 rows:


,date,covers,revenue,avg_check
0,2023-01-01,46.0,1653.23,35.94
1,2023-01-02,42.0,1161.42,27.65
2,2023-01-03,52.0,1523.81,29.30
3,2023-01-04,56.0,649.05,11.59
4,2023-01-05,56.0,885.16,15.81


---
## Phase B — Dimension Join (DuckDB)

For `dim_events`: multiple events can overlap on the same day (e.g. a fair and a match).
- Add sequential `event_id` to dim_events, update CSV
- In the SQL join, use `ROW_NUMBER() OVER (PARTITION BY date ORDER BY magnitude DESC)` to keep the highest-magnitude event per day
- Days with no event → `event_magnitude = 0`, `event_radius_km = NULL`

All dim tables have 731 rows with zero gaps, so LEFT JOINs are safe.

In [6]:
# ── Add event_id to dim_events (one id per event, not per row) ─────────────
# Key: event_name + year — identifies a unique event across its multi-day span
dim_events = pd.read_csv(EXT_PATH / 'dim_events.csv')

if 'event_id' not in dim_events.columns:
    dim_events['_year'] = pd.to_datetime(dim_events['date']).dt.year
    event_key = dim_events['event_name'].astype(str) + '_' + dim_events['_year'].astype(str)
    # Map each unique event key to a sequential integer
    key_to_id = {k: i+1 for i, k in enumerate(event_key.unique())}
    dim_events.insert(0, 'event_id', event_key.map(key_to_id))
    dim_events = dim_events.drop(columns=['_year'])
    dim_events.to_csv(EXT_PATH / 'dim_events.csv', index=False)
    print('event_id added (by event_name + year) and CSV updated.')
else:
    # Re-derive even if already present, to verify correctness
    dim_events['_year'] = pd.to_datetime(dim_events['date']).dt.year
    event_key = dim_events['event_name'].astype(str) + '_' + dim_events['_year'].astype(str)
    key_to_id = {k: i+1 for i, k in enumerate(event_key.unique())}
    dim_events['event_id'] = event_key.map(key_to_id)
    dim_events = dim_events.drop(columns=['_year'])
    dim_events.to_csv(EXT_PATH / 'dim_events.csv', index=False)
    print('event_id re-derived by event_name + year and CSV updated.')

print(f'dim_events: {dim_events.shape} — unique events: {dim_events["event_id"].nunique()}')
print(f'Columns: {dim_events.columns.tolist()}')
print()
# Sanity check: multi-day event should share the same event_id
sample = dim_events[dim_events['event_name'].str.contains('Salone', na=False)][['event_id','date','event_name']]
display(sample)


event_id re-derived by event_name + year and CSV updated.
dim_events: (204, 7) — unique events: 58
Columns: ['event_id', 'date', 'city', 'event_name', 'event_type', 'magnitude', 'radius_km']



,event_id,date,event_name
12,4,2023-04-18,Salone del Mobile 2023
13,4,2023-04-19,Salone del Mobile 2023
14,4,2023-04-20,Salone del Mobile 2023
15,4,2023-04-21,Salone del Mobile 2023
16,4,2023-04-22,Salone del Mobile 2023
17,4,2023-04-23,Salone del Mobile 2023
125,40,2024-04-16,Salone del Mobile 2024
126,40,2024-04-17,Salone del Mobile 2024
127,40,2024-04-18,Salone del Mobile 2024
128,40,2024-04-19,Salone del Mobile 2024


In [7]:
# ── Load all dimension tables ─────────────────────────────────────────────────
dim_calendar = pd.read_csv(EXT_PATH / 'dim_calendar.csv')
dim_weather  = pd.read_csv(EXT_PATH / 'dim_weather.csv')
dim_energy   = pd.read_csv(EXT_PATH / 'dim_energy.csv')

# Ensure date columns are strings for DuckDB matching
for df in [dim_calendar, dim_weather, dim_events, dim_energy]:
    df['date'] = df['date'].astype(str)

daily_pos['date'] = daily_pos['date'].dt.strftime('%Y-%m-%d')

print('Dim tables loaded:')
print(f'  dim_calendar : {dim_calendar.shape}')
print(f'  dim_weather  : {dim_weather.shape}')
print(f'  dim_events   : {dim_events.shape}')
print(f'  dim_energy   : {dim_energy.shape}')
print(f'  daily_pos    : {daily_pos.shape}')

Dim tables loaded:
  dim_calendar : (731, 7)
  dim_weather  : (731, 4)
  dim_events   : (204, 7)
  dim_energy   : (731, 3)
  daily_pos    : (731, 4)


In [8]:
# ── DuckDB join ───────────────────────────────────────────────────────────────
con = duckdb.connect()
con.register('daily_pos',    daily_pos)
con.register('dim_calendar', dim_calendar)
con.register('dim_weather',  dim_weather)
con.register('dim_events',   dim_events)
con.register('dim_energy',   dim_energy)

query = """
WITH events_ranked AS (
    -- Rank events per day by magnitude (highest first)
    SELECT
        date,
        event_id,
        event_name,
        event_type,
        magnitude  AS event_magnitude,
        radius_km  AS event_radius_km,
        ROW_NUMBER() OVER (PARTITION BY date ORDER BY magnitude DESC, event_id) AS rn
    FROM dim_events
),
events_daily AS (
    -- Keep only the dominant event per day
    SELECT date, event_id, event_name, event_type, event_magnitude, event_radius_km
    FROM events_ranked
    WHERE rn = 1
)
SELECT
    pos.date,

    -- POS aggregates
    pos.covers,
    pos.revenue,
    pos.avg_check,

    -- Calendar
    cal.is_weekend,
    cal.is_holiday,
    cal.is_ponte,
    cal.is_high_season,
    cal.season,

    -- Weather
    w.avg_temp,
    w.rain_mm,
    w.is_bad_weather,

    -- Events (0 if no event that day)
    COALESCE(ev.event_magnitude, 0)    AS event_magnitude,
    COALESCE(ev.event_radius_km, 0) AS event_radius_km,
    ev.event_name,
    ev.event_type,

    -- Energy
    en.pun_eur_mwh,
    en.energy_cost_index

FROM daily_pos pos
LEFT JOIN dim_calendar cal ON pos.date = cal.date
LEFT JOIN dim_weather  w   ON pos.date = w.date
LEFT JOIN events_daily ev  ON pos.date = ev.date
LEFT JOIN dim_energy   en  ON pos.date = en.date
ORDER BY pos.date
"""

daily_ts = con.execute(query).df()
con.close()

print(f'daily_timeseries shape: {daily_ts.shape}')
print(f'Date range: {daily_ts["date"].min()} → {daily_ts["date"].max()}')

daily_timeseries shape: (731, 18)
Date range: 2023-01-01 → 2024-12-31


---
## Phase C — Output Validation

In [9]:
# ── NaN check ────────────────────────────────────────────────────────────────
print('NaN per column:')
nan_counts = daily_ts.isna().sum()
print(nan_counts[nan_counts > 0].to_string() if nan_counts.any() else '  No NaN found ✓')

print()
print('First 5 rows:')
display(daily_ts.head())

print()
print('Descriptive stats (numeric columns):')
display(daily_ts.describe().round(2))

print()
print(f'Days with events: {(daily_ts["event_magnitude"] > 0).sum()} / {len(daily_ts)}')
print(f'event_type distribution:')
print(daily_ts['event_type'].value_counts(dropna=False).to_string())

NaN per column:
event_name    541
event_type    541

First 5 rows:


,date,covers,revenue,avg_check,is_weekend,is_holiday,is_ponte,is_high_season,season,avg_temp,rain_mm,is_bad_weather,event_magnitude,event_radius_km,event_name,event_type,pun_eur_mwh,energy_cost_index
0,2023-01-01,46.0,1653.23,35.94,True,True,False,False,winter,9.8,2.2,False,2,0,Ponte Capodanno,tourism_peak,193.14,369.12
1,2023-01-02,42.0,1161.42,27.65,False,False,False,False,winter,9.8,2.5,False,2,0,Ponte Capodanno,tourism_peak,202.04,386.12
2,2023-01-03,52.0,1523.81,29.30,False,False,False,False,winter,9.8,4.4,False,0,0,None,None,174.74,333.95
3,2023-01-04,56.0,649.05,11.59,False,False,False,False,winter,8.8,0.0,False,0,0,None,None,171.61,327.96
4,2023-01-05,56.0,885.16,15.81,False,False,False,False,winter,9.1,0.0,False,0,0,None,None,182.80,349.34



Descriptive stats (numeric columns):


,covers,revenue,avg_check,avg_temp,rain_mm,event_magnitude,event_radius_km,pun_eur_mwh,energy_cost_index
count,731.00,731.00,731.00,731.00,731.00,731.00,731.00,731.00,731.00
mean,54.14,2003.75,37.27,14.63,4.99,0.60,4.43,117.87,225.25
std,8.80,973.22,18.45,6.87,11.56,1.07,13.32,25.34,48.43
min,21.00,233.21,8.94,0.90,0.00,0.00,0.00,55.66,106.37
25%,48.00,1322.60,24.66,8.80,0.00,0.00,0.00,101.21,193.43
50%,54.00,1810.24,33.61,14.00,0.20,0.00,0.00,115.75,221.22
75%,60.00,2485.14,45.57,20.40,4.20,1.00,0.00,131.55,251.40
max,83.00,7216.53,160.87,30.00,104.80,3.00,50.00,204.11,390.08



Days with events: 190 / 731
event_type distribution:
event_type
None            541
fair             68
market           52
tourism_peak     30
sports           26
festival          7
concert           7


In [10]:
# ── Save ─────────────────────────────────────────────────────────────────────
output_path = PROCESSED / 'daily_timeseries.csv'
daily_ts.to_csv(output_path, index=False)
print(f'Saved: {output_path}')
print(f'Shape: {daily_ts.shape}')


Saved: /Users/giovannilopresti/Documents/Data Analytics/Progetti Risto/horeca_forecasting/data/processed/daily_timeseries.csv
Shape: (731, 18)
